In [3]:
import sys

print(sys.executable)
print(sys.version)

c:\Users\shafi\number-simplex-reproduction\.venv\Scripts\python.exe
3.13.15 (tags/v3.13.15:4061bc4, Aug  5 2026, 13:05:39) [MSC v.1944 64 bit (AMD64)]


In [4]:
import sys
import sklearn
import numpy as np
import pandas as pd
import scipy

print("Python executable:", sys.executable)
print("Python version:", sys.version)
print("scikit-learn:", sklearn.__version__)
print("NumPy:", np.__version__)
print("SciPy:", scipy.__version__)

Python executable: c:\Users\shafi\number-simplex-reproduction\.venv\Scripts\python.exe
Python version: 3.13.15 (tags/v3.13.15:4061bc4, Aug  5 2026, 13:05:39) [MSC v.1944 64 bit (AMD64)]
scikit-learn: 1.9.1
NumPy: 2.5.3
SciPy: 1.18.1


In [5]:
import sys
from pathlib import Path

cwd = Path.cwd()

if (cwd / "src").exists():
    PROJECT_ROOT = cwd

elif (cwd.parent / "src").exists():
    PROJECT_ROOT = cwd.parent

else:
    raise RuntimeError(
        "Could not find the src folder. "
        f"Current directory is: {cwd}"
    )

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

print("Current directory:", cwd)
print("Project root:", PROJECT_ROOT)
print("src exists:", (PROJECT_ROOT / "src").exists())

Current directory: c:\Users\shafi\number-simplex-reproduction\notebooks1
Project root: c:\Users\shafi\number-simplex-reproduction
src exists: True


In [6]:
from src.poisson_population_analysis import build_temporal_population

subject = "YFF"
bin_size = 100

data = build_temporal_population(
    subject=subject,
    root=PROJECT_ROOT,
    bin_size=bin_size
)

X = data["X"]
y = data["y"]
groups = data["groups"]
operands = data["operands"]
presentations = data["presentations"]
feature_map = data["feature_map"]
neurons = data["neurons"]
bin_edges = data["bin_edges"]

print("Subject:", subject)
print("X shape:", X.shape)
print("y shape:", y.shape)
print("groups shape:", groups.shape)
print("Number of neurons:", len(neurons))
print("Classes:", np.unique(y))
print("Bin edges:", bin_edges)

print("\nClass counts:")
print(pd.Series(y).value_counts().sort_index())

Subject: YFF
X shape: (109, 333)
y shape: (109,)
groups shape: (109,)
Number of neurons: 37
Classes: [1 2 3 4 5 6 7 8 9]
Bin edges: [ 50. 150. 250. 350. 450. 550. 650. 750. 850. 950.]

Class counts:
1    10
2    15
3    11
4     7
5     7
6    16
7    17
8    12
9    14
Name: count, dtype: int64


In [7]:
from sklearn.model_selection import StratifiedGroupKFold

outer_cv = StratifiedGroupKFold(
    n_splits=5,
    shuffle=True,
    random_state=42
)

outer_splits = list(
    outer_cv.split(X, y, groups=groups)
)

print("Number of outer folds:", len(outer_splits))

for fold, (train_idx, test_idx) in enumerate(outer_splits, start=1):
    
    train_groups = np.unique(groups[train_idx])
    test_groups = np.unique(groups[test_idx])
    
    print(f"\nFold {fold}")
    print("  train presentations:", len(train_idx))
    print("  test presentations :", len(test_idx))
    print("  train trials       :", len(train_groups))
    print("  test trials        :", len(test_groups))
    print("  overlap            :", len(
        np.intersect1d(train_groups, test_groups)
    ))

Number of outer folds: 5

Fold 1
  train presentations: 87
  test presentations : 22
  train trials       : 66
  test trials        : 16
  overlap            : 0

Fold 2
  train presentations: 87
  test presentations : 22
  train trials       : 65
  test trials        : 17
  overlap            : 0

Fold 3
  train presentations: 87
  test presentations : 22
  train trials       : 66
  test trials        : 16
  overlap            : 0

Fold 4
  train presentations: 87
  test presentations : 22
  train trials       : 65
  test trials        : 17
  overlap            : 0

Fold 5
  train presentations: 88
  test presentations : 21
  train trials       : 66
  test trials        : 16
  overlap            : 0


In [8]:
fold_assignment = np.full(len(y), -1, dtype=int)

for fold, (_, test_idx) in enumerate(outer_splits, start=1):
    fold_assignment[test_idx] = fold

cv_table = pd.DataFrame({
    "presentation_index": np.arange(len(y)),
    "number": y,
    "trial": groups,
    "outer_fold": fold_assignment,
})

save_dir = PROJECT_ROOT / "tables1" / "cv_splits"
save_dir.mkdir(parents=True, exist_ok=True)

save_path = save_dir / "YFF_outer_folds_sklearn_1_9_1.csv"
cv_table.to_csv(save_path, index=False)

print("Saved to:")
print(save_path)

print("\nFold counts:")
print(cv_table["outer_fold"].value_counts().sort_index())

Saved to:
c:\Users\shafi\number-simplex-reproduction\tables1\cv_splits\YFF_outer_folds_sklearn_1_9_1.csv

Fold counts:
outer_fold
1    22
2    22
3    22
4    22
5    21
Name: count, dtype: int64


In [9]:
def gamma_sufficient_stats(X_train, y_train, classes):
    """
    Compute the sufficient statistics needed for the
    Gamma-Poisson posterior.

    Returns
    -------
    sum_counts : shape (n_classes, n_features)
        Total spike count for each class and feature.

    n_class : shape (n_classes,)
        Number of training presentations in each class.
    """

    n_classes = len(classes)
    n_features = X_train.shape[1]

    sum_counts = np.zeros((n_classes, n_features))
    n_class = np.zeros(n_classes)

    for class_index, k in enumerate(classes):

        X_k = X_train[y_train == k]

        n_class[class_index] = len(X_k)

        sum_counts[class_index] = X_k.sum(axis=0)

    return sum_counts, n_class

In [10]:
classes = np.arange(1, 10)

train_idx, test_idx = outer_splits[0]

X_train = X[train_idx]
y_train = y[train_idx]

X_test = X[test_idx]
y_test = y[test_idx]

sum_counts, n_class = gamma_sufficient_stats(
    X_train,
    y_train,
    classes
)

print("X_train shape:", X_train.shape)
print("X_test shape:", X_test.shape)

print("\nsum_counts shape:", sum_counts.shape)
print("n_class shape:", n_class.shape)

print("\nTraining presentations per number:")
for k, n in zip(classes, n_class):
    print(f"Number {k}: {int(n)}")

X_train shape: (87, 333)
X_test shape: (22, 333)

sum_counts shape: (9, 333)
n_class shape: (9,)

Training presentations per number:
Number 1: 8
Number 2: 12
Number 3: 8
Number 4: 5
Number 5: 6
Number 6: 12
Number 7: 14
Number 8: 10
Number 9: 12


In [11]:
from scipy.special import gammaln

def predict_gamma_poisson_from_stats(
    X_test,
    sum_counts,
    n_class,
    classes,
    alpha,
    beta,
):
    """
    Gamma-Poisson posterior predictive classifier.

    alpha : scalar Gamma prior shape
    beta  : scalar Gamma prior rate
    """

    # Posterior parameters
    a_post = alpha + sum_counts
    b_post = beta + n_class[:, None]

    # One score for every test presentation × candidate class
    log_scores = np.zeros(
        (X_test.shape[0], len(classes))
    )

    for class_index, k in enumerate(classes):

        a = a_post[class_index]
        b = b_post[class_index]

        log_scores[:, class_index] = (
            gammaln(X_test + a)
            - gammaln(a)
            + a * np.log(b / (b + 1))
            + X_test * np.log(1 / (b + 1))
        ).sum(axis=1)

    predictions = classes[
        np.argmax(log_scores, axis=1)
    ]

    return predictions

In [12]:
alpha_test = 1.0
beta_test = 1.0

pred_test = predict_gamma_poisson_from_stats(
    X_test=X_test,
    sum_counts=sum_counts,
    n_class=n_class,
    classes=classes,
    alpha=alpha_test,
    beta=beta_test,
)

print("Predictions shape:", pred_test.shape)
print("Predictions:")
print(pred_test)

print("\nTrue labels:")
print(y_test)

print("\nUnique predicted classes:")
print(np.unique(pred_test))

Predictions shape: (22,)
Predictions:
[9 7 8 6 7 7 2 1 2 8 4 6 8 7 7 7 8 2 2 8 2 2]

True labels:
[5 6 1 6 6 8 1 7 4 3 8 6 7 7 9 9 2 2 4 3 2 3]

Unique predicted classes:
[1 2 4 6 7 8 9]


In [13]:
prior_means_nested = np.geomspace(
    0.03,
    0.4,
    10
)

prior_strengths_nested = np.geomspace(
    10,
    250,
    12
)

print("Prior means:")
print(prior_means_nested)

print("\nPrior strengths:")
print(prior_strengths_nested)

print(
    "\nTotal candidates:",
    len(prior_means_nested)
    * len(prior_strengths_nested)
)

Prior means:
[0.03       0.04000502 0.05334671 0.07113787 0.09486238 0.12649904
 0.16868653 0.22494358 0.29996239 0.4       ]

Prior strengths:
[ 10.          13.39940319  17.95440059  24.05782525  32.23605005
  43.19438319  57.87789559  77.55292588 103.91629226 139.24162981
 186.57547387 250.        ]

Total candidates: 120


In [14]:
from sklearn.model_selection import StratifiedGroupKFold

# Outer Fold 1
outer_fold = 1
outer_train_idx, outer_test_idx = outer_splits[0]

X_outer_train = X[outer_train_idx]
y_outer_train = y[outer_train_idx]
groups_outer_train = groups[outer_train_idx]

X_outer_test = X[outer_test_idx]
y_outer_test = y[outer_test_idx]

inner_cv = StratifiedGroupKFold(
    n_splits=4,
    shuffle=True,
    random_state=100 + outer_fold
)

inner_splits = list(
    inner_cv.split(
        X_outer_train,
        y_outer_train,
        groups=groups_outer_train
    )
)

print("Outer training shape:", X_outer_train.shape)
print("Outer test shape:", X_outer_test.shape)
print("Number of inner folds:", len(inner_splits))

for inner_fold, (inner_train_idx, inner_val_idx) in enumerate(
    inner_splits,
    start=1
):
    train_groups = np.unique(
        groups_outer_train[inner_train_idx]
    )
    val_groups = np.unique(
        groups_outer_train[inner_val_idx]
    )

    print(f"\nInner fold {inner_fold}")
    print("  train presentations:", len(inner_train_idx))
    print("  validation presentations:", len(inner_val_idx))
    print(
        "  group overlap:",
        len(np.intersect1d(train_groups, val_groups))
    )

Outer training shape: (87, 333)
Outer test shape: (22, 333)
Number of inner folds: 4

Inner fold 1
  train presentations: 65
  validation presentations: 22
  group overlap: 0

Inner fold 2
  train presentations: 66
  validation presentations: 21
  group overlap: 0

Inner fold 3
  train presentations: 66
  validation presentations: 21
  group overlap: 0

Inner fold 4
  train presentations: 64
  validation presentations: 23
  group overlap: 0


In [15]:
inner_data = []

for inner_train_idx, inner_val_idx in inner_splits:

    X_inner_train = X_outer_train[inner_train_idx]
    y_inner_train = y_outer_train[inner_train_idx]

    X_inner_val = X_outer_train[inner_val_idx]
    y_inner_val = y_outer_train[inner_val_idx]

    sum_counts_inner, n_class_inner = gamma_sufficient_stats(
        X_inner_train,
        y_inner_train,
        classes
    )

    inner_data.append(
        (
            X_inner_val,
            y_inner_val,
            sum_counts_inner,
            n_class_inner
        )
    )

print("Prepared inner folds:", len(inner_data))

for i, (
    X_val,
    y_val,
    sum_counts_inner,
    n_class_inner
) in enumerate(inner_data, start=1):

    print(
        f"Fold {i}:",
        "validation =", X_val.shape,
        "sum_counts =", sum_counts_inner.shape
    )

Prepared inner folds: 4
Fold 1: validation = (22, 333) sum_counts = (9, 333)
Fold 2: validation = (21, 333) sum_counts = (9, 333)
Fold 3: validation = (21, 333) sum_counts = (9, 333)
Fold 4: validation = (23, 333) sum_counts = (9, 333)


In [16]:
from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score
)

candidate_results = []

for m in prior_means_nested:

    for s in prior_strengths_nested:

        alpha = m * s
        beta = s

        inner_true = []
        inner_pred = []

        for (
            X_val,
            y_val,
            sum_counts_inner,
            n_class_inner
        ) in inner_data:

            pred = predict_gamma_poisson_from_stats(
                X_test=X_val,
                sum_counts=sum_counts_inner,
                n_class=n_class_inner,
                classes=classes,
                alpha=alpha,
                beta=beta
            )

            inner_true.append(y_val)
            inner_pred.append(pred)

        inner_true = np.concatenate(inner_true)
        inner_pred = np.concatenate(inner_pred)

        candidate_results.append({
            "prior_mean": m,
            "prior_strength": s,
            "alpha": alpha,
            "beta": beta,
            "inner_accuracy": accuracy_score(
                inner_true,
                inner_pred
            ),
            "inner_balanced_accuracy":
                balanced_accuracy_score(
                    inner_true,
                    inner_pred
                )
        })

candidate_df = pd.DataFrame(candidate_results)

print("Number of candidates:", len(candidate_df))

Number of candidates: 120


In [17]:
best = (
    candidate_df
    .sort_values(
        [
            "inner_balanced_accuracy",
            "inner_accuracy"
        ],
        ascending=False
    )
    .iloc[0]
)

print(best)

prior_mean                  0.126499
prior_strength             13.399403
alpha                       1.695012
beta                       13.399403
inner_accuracy              0.160920
inner_balanced_accuracy     0.121693
Name: 61, dtype: float64


In [18]:
best_alpha = best["alpha"]
best_beta = best["beta"]

outer_sum_counts, outer_n_class = gamma_sufficient_stats(
    X_outer_train,
    y_outer_train,
    classes
)

print("Posterior sufficient statistics:")
print("sum_counts shape:", outer_sum_counts.shape)
print("n_class:", outer_n_class)

print("\nSelected prior:")
print("m     =", best["prior_mean"])
print("tau   =", best["prior_strength"])
print("alpha =", best_alpha)
print("beta  =", best_beta)

Posterior sufficient statistics:
sum_counts shape: (9, 333)
n_class: [ 8. 12.  8.  5.  6. 12. 14. 10. 12.]

Selected prior:
m     = 0.12649903703900578
tau   = 13.399403191028387
alpha = 1.6950116005624722
beta  = 13.399403191028387


In [19]:
gamma_pred_fold1 = predict_gamma_poisson_from_stats(
    X_test=X_outer_test,
    sum_counts=outer_sum_counts,
    n_class=outer_n_class,
    classes=classes,
    alpha=best_alpha,
    beta=best_beta
)

gamma_acc_fold1 = accuracy_score(
    y_outer_test,
    gamma_pred_fold1
)

gamma_bal_fold1 = balanced_accuracy_score(
    y_outer_test,
    gamma_pred_fold1
)

print("Gamma Fold-1 accuracy:",
      gamma_acc_fold1)

print("Gamma Fold-1 balanced accuracy:",
      gamma_bal_fold1)

print(
    "Correct:",
    np.sum(gamma_pred_fold1 == y_outer_test),
    "/",
    len(y_outer_test)
)

Gamma Fold-1 accuracy: 0.13636363636363635
Gamma Fold-1 balanced accuracy: 0.1111111111111111
Correct: 3 / 22


In [20]:
from src.poisson_naive_bayes import (
    fit_poisson_nb,
    predict_poisson_nb
)

In [21]:
def run_nested_gamma_subject(
    X,
    y,
    groups,
    classes,
    prior_means,
    prior_strengths,
):

    outer_cv = StratifiedGroupKFold(
        n_splits=5,
        shuffle=True,
        random_state=42
    )

    fold_results = []

    gamma_true_all = []
    gamma_pred_all = []

    mle_true_all = []
    mle_pred_all = []

    for outer_fold, (outer_train_idx, outer_test_idx) in enumerate(
        outer_cv.split(X, y, groups=groups),
        start=1
    ):

        print(f"Outer fold {outer_fold}")

        X_outer_train = X[outer_train_idx]
        y_outer_train = y[outer_train_idx]
        groups_outer_train = groups[outer_train_idx]

        X_outer_test = X[outer_test_idx]
        y_outer_test = y[outer_test_idx]

        # -----------------------------
        # Inner CV
        # -----------------------------

        inner_cv = StratifiedGroupKFold(
            n_splits=4,
            shuffle=True,
            random_state=100 + outer_fold
        )

        inner_splits = list(
            inner_cv.split(
                X_outer_train,
                y_outer_train,
                groups=groups_outer_train
            )
        )

        # Precompute sufficient statistics
        inner_data = []

        for inner_train_idx, inner_val_idx in inner_splits:

            X_inner_train = X_outer_train[inner_train_idx]
            y_inner_train = y_outer_train[inner_train_idx]

            X_inner_val = X_outer_train[inner_val_idx]
            y_inner_val = y_outer_train[inner_val_idx]

            sum_counts, n_class = gamma_sufficient_stats(
                X_inner_train,
                y_inner_train,
                classes
            )

            inner_data.append(
                (
                    X_inner_val,
                    y_inner_val,
                    sum_counts,
                    n_class
                )
            )

        # -----------------------------
        # Search Gamma prior
        # -----------------------------

        candidate_results = []

        for m in prior_means:

            for s in prior_strengths:

                alpha = m * s
                beta = s

                inner_true = []
                inner_pred = []

                for (
                    X_val,
                    y_val,
                    sum_counts,
                    n_class
                ) in inner_data:

                    pred = predict_gamma_poisson_from_stats(
                        X_val,
                        sum_counts,
                        n_class,
                        classes,
                        alpha,
                        beta
                    )

                    inner_true.append(y_val)
                    inner_pred.append(pred)

                inner_true = np.concatenate(inner_true)
                inner_pred = np.concatenate(inner_pred)

                candidate_results.append(
                    {
                        "prior_mean": m,
                        "prior_strength": s,
                        "alpha": alpha,
                        "beta": beta,
                        "inner_accuracy":
                            accuracy_score(
                                inner_true,
                                inner_pred
                            ),
                        "inner_balanced_accuracy":
                            balanced_accuracy_score(
                                inner_true,
                                inner_pred
                            )
                    }
                )

        candidate_df = pd.DataFrame(candidate_results)

        # Balanced accuracy primary,
        # ordinary accuracy tie-breaker
        best = (
            candidate_df
            .sort_values(
                [
                    "inner_balanced_accuracy",
                    "inner_accuracy"
                ],
                ascending=False
            )
            .iloc[0]
        )

        best_alpha = best["alpha"]
        best_beta = best["beta"]

        # -----------------------------
        # Fit Gamma-Poisson using
        # complete outer training set
        # -----------------------------

        outer_sum, outer_n = gamma_sufficient_stats(
            X_outer_train,
            y_outer_train,
            classes
        )

        gamma_pred = predict_gamma_poisson_from_stats(
            X_outer_test,
            outer_sum,
            outer_n,
            classes,
            best_alpha,
            best_beta
        )

        # -----------------------------
        # MLE Poisson NB baseline
        # same outer split
        # -----------------------------

        mle_model = fit_poisson_nb(
            X_outer_train,
            y_outer_train,
            classes=classes,
            uniform_prior=True
        )

        mle_pred = predict_poisson_nb(
            mle_model,
            X_outer_test
        )

        # -----------------------------
        # Fold metrics
        # -----------------------------

        gamma_acc = accuracy_score(
            y_outer_test,
            gamma_pred
        )

        gamma_bal = balanced_accuracy_score(
            y_outer_test,
            gamma_pred
        )

        mle_acc = accuracy_score(
            y_outer_test,
            mle_pred
        )

        mle_bal = balanced_accuracy_score(
            y_outer_test,
            mle_pred
        )

        print(
            f"  prior mean={best['prior_mean']:.4f}, "
            f"strength={best['prior_strength']:.2f}"
        )

        print(
            f"  Gamma={gamma_acc:.4f}, "
            f"MLE={mle_acc:.4f}"
        )

        fold_results.append(
            {
                "outer_fold": outer_fold,
                "prior_mean": best["prior_mean"],
                "prior_strength": best["prior_strength"],
                "alpha": best_alpha,
                "beta": best_beta,
                "gamma_accuracy": gamma_acc,
                "gamma_balanced_accuracy": gamma_bal,
                "mle_accuracy": mle_acc,
                "mle_balanced_accuracy": mle_bal,
            }
        )

        gamma_true_all.append(y_outer_test)
        gamma_pred_all.append(gamma_pred)

        mle_true_all.append(y_outer_test)
        mle_pred_all.append(mle_pred)

    # -----------------------------
    # Pool all held-out predictions
    # -----------------------------

    gamma_true_all = np.concatenate(gamma_true_all)
    gamma_pred_all = np.concatenate(gamma_pred_all)

    mle_true_all = np.concatenate(mle_true_all)
    mle_pred_all = np.concatenate(mle_pred_all)

    summary = {
        "gamma_accuracy":
            accuracy_score(
                gamma_true_all,
                gamma_pred_all
            ),

        "gamma_balanced_accuracy":
            balanced_accuracy_score(
                gamma_true_all,
                gamma_pred_all
            ),

        "mle_accuracy":
            accuracy_score(
                mle_true_all,
                mle_pred_all
            ),

        "mle_balanced_accuracy":
            balanced_accuracy_score(
                mle_true_all,
                mle_pred_all
            ),

        "gamma_correct":
            int(np.sum(
                gamma_true_all == gamma_pred_all
            )),

        "mle_correct":
            int(np.sum(
                mle_true_all == mle_pred_all
            )),

        "n_presentations":
            len(gamma_true_all),
    }

    return summary, pd.DataFrame(fold_results)

In [22]:
summary_yff, folds_yff = run_nested_gamma_subject(
    X=X,
    y=y,
    groups=groups,
    classes=classes,
    prior_means=prior_means_nested,
    prior_strengths=prior_strengths_nested
)

Outer fold 1
  prior mean=0.1265, strength=13.40
  Gamma=0.1364, MLE=0.1364
Outer fold 2
  prior mean=0.2249, strength=10.00
  Gamma=0.2727, MLE=0.1818
Outer fold 3
  prior mean=0.2249, strength=13.40
  Gamma=0.1364, MLE=0.1364
Outer fold 4
  prior mean=0.4000, strength=10.00
  Gamma=0.1818, MLE=0.2273
Outer fold 5
  prior mean=0.3000, strength=10.00
  Gamma=0.2381, MLE=0.2381


In [23]:
print("\nYFF SUMMARY")
print(summary_yff)

print("\nFOLD RESULTS")
display(folds_yff)


YFF SUMMARY
{'gamma_accuracy': 0.1926605504587156, 'gamma_balanced_accuracy': 0.14899237472766882, 'mle_accuracy': 0.1834862385321101, 'mle_balanced_accuracy': 0.15303207141442435, 'gamma_correct': 21, 'mle_correct': 20, 'n_presentations': 109}

FOLD RESULTS


,outer_fold,prior_mean,prior_strength,alpha,beta,gamma_accuracy,gamma_balanced_accuracy,mle_accuracy,mle_balanced_accuracy
0,1,0.126499,13.399403,1.695012,13.399403,0.136364,0.111111,0.136364,0.092593
1,2,0.224944,10.000000,2.249436,10.000000,0.272727,0.185185,0.181818,0.148148
2,3,0.224944,13.399403,3.014110,13.399403,0.136364,0.111111,0.136364,0.111111
3,4,0.400000,10.000000,4.000000,10.000000,0.181818,0.129630,0.227273,0.240741
4,5,0.299962,10.000000,2.999624,10.000000,0.238095,0.185185,0.238095,0.185185


In [24]:
results_dir = PROJECT_ROOT / "tables1" / "gamma_poisson"
results_dir.mkdir(parents=True, exist_ok=True)

# Save fold-level results
folds_yff.to_csv(
    results_dir / "YFF_nested_gamma_fold_results.csv",
    index=False
)

# Save subject summary
pd.DataFrame([summary_yff]).to_csv(
    results_dir / "YFF_nested_gamma_summary.csv",
    index=False
)

print("Saved YFF results to:")
print(results_dir)

Saved YFF results to:
c:\Users\shafi\number-simplex-reproduction\tables1\gamma_poisson


In [25]:
subjects = [
    "YFF",
    "YFI",
    "YFJ",
    "YFK",
    "YFL",
    "YFM",
    "YFP",
    "YFR",
    "YFS",
    "YFT",
    "YFU",
]

print("Number of subjects:", len(subjects))

Number of subjects: 11


In [26]:
import time

all_subject_summaries = []
all_subject_fold_results = []

for subject in subjects:

    print("\n" + "=" * 60)
    print("SUBJECT:", subject)
    print("=" * 60)

    # Build temporal population
    data_sub = build_temporal_population(
        subject=subject,
        root=PROJECT_ROOT,
        bin_size=100
    )

    X_sub = data_sub["X"]
    y_sub = data_sub["y"]
    groups_sub = data_sub["groups"]

    print("X:", X_sub.shape)

    start = time.perf_counter()

    summary, fold_df = run_nested_gamma_subject(
        X_sub,
        y_sub,
        groups_sub,
        classes,
        prior_means_nested,
        prior_strengths_nested,
    )

    elapsed = time.perf_counter() - start

    summary["subject"] = subject
    summary["n_features"] = X_sub.shape[1]
    summary["elapsed_seconds"] = elapsed

    fold_df["subject"] = subject

    all_subject_summaries.append(summary)
    all_subject_fold_results.append(fold_df)

    print("\nSubject summary:")
    print(summary)

    print(
        f"Elapsed: {elapsed / 60:.2f} minutes"
    )


SUBJECT: YFF
X: (109, 333)
Outer fold 1
  prior mean=0.1265, strength=13.40
  Gamma=0.1364, MLE=0.1364
Outer fold 2
  prior mean=0.2249, strength=10.00
  Gamma=0.2727, MLE=0.1818
Outer fold 3
  prior mean=0.2249, strength=13.40
  Gamma=0.1364, MLE=0.1364
Outer fold 4
  prior mean=0.4000, strength=10.00
  Gamma=0.1818, MLE=0.2273
Outer fold 5
  prior mean=0.3000, strength=10.00
  Gamma=0.2381, MLE=0.2381

Subject summary:
{'gamma_accuracy': 0.1926605504587156, 'gamma_balanced_accuracy': 0.14899237472766882, 'mle_accuracy': 0.1834862385321101, 'mle_balanced_accuracy': 0.15303207141442435, 'gamma_correct': 21, 'mle_correct': 20, 'n_presentations': 109, 'subject': 'YFF', 'n_features': 333, 'elapsed_seconds': 7.17647860001307}
Elapsed: 0.12 minutes

SUBJECT: YFI
X: (104, 261)
Outer fold 1
  prior mean=0.0300, strength=10.00
  Gamma=0.1905, MLE=0.1905
Outer fold 2
  prior mean=0.0300, strength=10.00
  Gamma=0.2500, MLE=0.2500
Outer fold 3
  prior mean=0.0300, strength=10.00
  Gamma=0.2500, 

c:\Users\shafi\number-simplex-reproduction\.venv\Lib\site-packages\sklearn\model_selection\_split.py:1036: UserWarning: The least populated class in y has only 4 members, which is less than n_splits=5.
  warnings.warn(
c:\Users\shafi\number-simplex-reproduction\.venv\Lib\site-packages\sklearn\model_selection\_split.py:1036: UserWarning: The least populated class in y has only 3 members, which is less than n_splits=4.
  warnings.warn(


  prior mean=0.0300, strength=43.19
  Gamma=0.1667, MLE=0.3333
Outer fold 2


c:\Users\shafi\number-simplex-reproduction\.venv\Lib\site-packages\sklearn\model_selection\_split.py:1036: UserWarning: The least populated class in y has only 3 members, which is less than n_splits=4.
  warnings.warn(


  prior mean=0.4000, strength=10.00
  Gamma=0.2609, MLE=0.1739
Outer fold 3


...

  prior mean=0.0533, strength=10.00
  Gamma=0.1304, MLE=0.0870
Outer fold 3


c:\Users\shafi\number-simplex-reproduction\.venv\Lib\site-packages\sklearn\model_selection\_split.py:1036: UserWarning: The least populated class in y has only 3 members, which is less than n_splits=4.
  warnings.warn(


  prior mean=0.0300, strength=139.24
  Gamma=0.1304, MLE=0.1739
Outer fold 4


c:\Users\shafi\number-simplex-reproduction\.venv\Lib\site-packages\sklearn\model_selection\_split.py:1036: UserWarning: The least populated class in y has only 3 members, which is less than n_splits=4.
  warnings.warn(


  prior mean=0.0711, strength=17.95
  Gamma=0.1364, MLE=0.1818
Outer fold 5
  prior mean=0.0400, strength=10.00
  Gamma=0.0909, MLE=0.0909

Subject summary:
{'gamma_accuracy': 0.12280701754385964, 'gamma_balanced_accuracy': 0.08763329893360854, 'mle_accuracy': 0.14035087719298245, 'mle_balanced_accuracy': 0.10128876111848248, 'gamma_correct': 14, 'mle_correct': 16, 'n_presentations': 114, 'subject': 'YFM', 'n_features': 549, 'elapsed_seconds': 10.074592499993742}
Elapsed: 0.17 minutes

SUBJECT: YFP
X: (158, 387)
Outer fold 1
  prior mean=0.4000, strength=43.19
  Gamma=0.1613, MLE=0.1613
Outer fold 2
  prior mean=0.3000, strength=10.00
  Gamma=0.0625, MLE=0.0625
Outer fold 3
  prior mean=0.3000, strength=13.40
  Gamma=0.0938, MLE=0.1250
Outer fold 4
  prior mean=0.1265, strength=10.00
  Gamma=0.2188, MLE=0.1250
Outer fold 5
  prior mean=0.2249, strength=13.40
  Gamma=0.2258, MLE=0.1935

Subject summary:
{'gamma_accuracy': 0.1518987341772152, 'gamma_balanced_accuracy': 0.1055796055796055

In [27]:
summary_all = pd.DataFrame(
    all_subject_summaries
)

folds_all = pd.concat(
    all_subject_fold_results,
    ignore_index=True
)

display(summary_all)

,gamma_accuracy,gamma_balanced_accuracy,mle_accuracy,mle_balanced_accuracy,gamma_correct,mle_correct,n_presentations,subject,n_features,elapsed_seconds
0,0.192661,0.148992,0.183486,0.153032,21,20,109,YFF,333,7.176479
1,0.221154,0.111111,0.221154,0.150022,23,23,104,YFI,261,6.964034
2,0.166667,0.118040,0.192982,0.152883,19,22,114,YFJ,405,6.002950
3,0.166667,0.125762,0.219298,0.173736,19,25,114,YFK,396,7.425998
4,0.157895,0.109545,0.149123,0.109250,18,17,114,YFL,513,9.082240
5,0.122807,0.087633,0.140351,0.101289,14,16,114,YFM,549,10.074592
6,0.151899,0.105580,0.132911,0.100632,24,21,158,YFP,387,8.406508
7,0.112782,0.102472,0.112782,0.101830,30,30,266,YFR,585,18.256939
8,0.106061,0.097012,0.128788,0.123494,28,34,264,YFS,531,14.368804
9,0.136268,0.126056,0.136268,0.131361,65,65,477,YFT,468,25.305612


In [28]:
summary_display = summary_all[
    [
        "subject",
        "n_presentations",
        "n_features",
        "gamma_accuracy",
        "gamma_balanced_accuracy",
        "gamma_correct",
        "mle_accuracy",
        "mle_balanced_accuracy",
        "mle_correct",
    ]
]

display(summary_display)

,subject,n_presentations,n_features,gamma_accuracy,gamma_balanced_accuracy,gamma_correct,mle_accuracy,mle_balanced_accuracy,mle_correct
0,YFF,109,333,0.192661,0.148992,21,0.183486,0.153032,20
1,YFI,104,261,0.221154,0.111111,23,0.221154,0.150022,23
2,YFJ,114,405,0.166667,0.118040,19,0.192982,0.152883,22
3,YFK,114,396,0.166667,0.125762,19,0.219298,0.173736,25
4,YFL,114,513,0.157895,0.109545,18,0.149123,0.109250,17
5,YFM,114,549,0.122807,0.087633,14,0.140351,0.101289,16
6,YFP,158,387,0.151899,0.105580,24,0.132911,0.100632,21
7,YFR,266,585,0.112782,0.102472,30,0.112782,0.101830,30
8,YFS,264,531,0.106061,0.097012,28,0.128788,0.123494,34
9,YFT,477,468,0.136268,0.126056,65,0.136268,0.131361,65


In [29]:
summary_all.to_csv(
    results_dir / "all_subjects_nested_gamma_summary.csv",
    index=False
)

folds_all.to_csv(
    results_dir / "all_subjects_nested_gamma_fold_results.csv",
    index=False
)

print("Saved:")
print(
    results_dir /
    "all_subjects_nested_gamma_summary.csv"
)

print(
    results_dir /
    "all_subjects_nested_gamma_fold_results.csv"
)

Saved:
c:\Users\shafi\number-simplex-reproduction\tables1\gamma_poisson\all_subjects_nested_gamma_summary.csv
c:\Users\shafi\number-simplex-reproduction\tables1\gamma_poisson\all_subjects_nested_gamma_fold_results.csv


In [30]:
print(
    "Mean Gamma accuracy:",
    summary_all["gamma_accuracy"].mean()
)

print(
    "Mean Gamma balanced accuracy:",
    summary_all["gamma_balanced_accuracy"].mean()
)

print(
    "Mean MLE accuracy:",
    summary_all["mle_accuracy"].mean()
)

print(
    "Mean MLE balanced accuracy:",
    summary_all["mle_balanced_accuracy"].mean()
)

Mean Gamma accuracy: 0.1494700785602819
Mean Gamma balanced accuracy: 0.11145924526448975
Mean MLE accuracy: 0.15695052558520278
Mean MLE balanced accuracy: 0.12758451334669166


In [31]:
def select_full_data_prior(
    X,
    y,
    groups,
    classes,
    prior_means,
    prior_strengths,
):

    cv = StratifiedGroupKFold(
        n_splits=5,
        shuffle=True,
        random_state=42
    )

    splits = list(
        cv.split(X, y, groups=groups)
    )

    fold_data = []

    for train_idx, val_idx in splits:

        X_train = X[train_idx]
        y_train = y[train_idx]

        X_val = X[val_idx]
        y_val = y[val_idx]

        sum_counts, n_class = gamma_sufficient_stats(
            X_train,
            y_train,
            classes
        )

        fold_data.append(
            (
                X_val,
                y_val,
                sum_counts,
                n_class
            )
        )

    results = []

    for m in prior_means:

        for s in prior_strengths:

            alpha = m * s
            beta = s

            all_true = []
            all_pred = []

            for (
                X_val,
                y_val,
                sum_counts,
                n_class
            ) in fold_data:

                pred = predict_gamma_poisson_from_stats(
                    X_val,
                    sum_counts,
                    n_class,
                    classes,
                    alpha,
                    beta
                )

                all_true.append(y_val)
                all_pred.append(pred)

            all_true = np.concatenate(all_true)
            all_pred = np.concatenate(all_pred)

            results.append({
                "prior_mean": m,
                "prior_strength": s,
                "alpha": alpha,
                "beta": beta,
                "accuracy": accuracy_score(
                    all_true,
                    all_pred
                ),
                "balanced_accuracy":
                    balanced_accuracy_score(
                        all_true,
                        all_pred
                    )
            })

    results_df = pd.DataFrame(results)

    best = (
        results_df
        .sort_values(
            [
                "balanced_accuracy",
                "accuracy"
            ],
            ascending=False
        )
        .iloc[0]
    )

    return best, results_df

In [32]:
def build_posterior_lambda_matrix(
    X,
    y,
    classes,
    alpha,
    beta
):

    sum_counts, n_class = gamma_sufficient_stats(
        X,
        y,
        classes
    )

    lambda_matrix = (
        alpha + sum_counts
    ) / (
        beta + n_class[:, None]
    )

    return lambda_matrix

In [33]:
lambda_dir = (
    PROJECT_ROOT
    / "tables1"
    / "gamma_poisson"
    / "lambda_matrices"
)

lambda_dir.mkdir(
    parents=True,
    exist_ok=True
)

lambda_matrices = {}
lambda_metadata = []

for subject in subjects:

    print("\n" + "=" * 60)
    print("SUBJECT:", subject)
    print("=" * 60)

    data_sub = build_temporal_population(
        subject=subject,
        root=PROJECT_ROOT,
        bin_size=100
    )

    X_sub = data_sub["X"]
    y_sub = data_sub["y"]
    groups_sub = data_sub["groups"]

    # ---------------------------------
    # Select one prior using full-data CV
    # ---------------------------------

    best_prior, _ = select_full_data_prior(
        X=X_sub,
        y=y_sub,
        groups=groups_sub,
        classes=classes,
        prior_means=prior_means_nested,
        prior_strengths=prior_strengths_nested
    )

    alpha = best_prior["alpha"]
    beta = best_prior["beta"]

    # ---------------------------------
    # Fit posterior using ALL data
    # ---------------------------------

    Lambda = build_posterior_lambda_matrix(
        X=X_sub,
        y=y_sub,
        classes=classes,
        alpha=alpha,
        beta=beta
    )

    lambda_matrices[subject] = Lambda

    print("Lambda shape:", Lambda.shape)
    print(
        f"m={best_prior['prior_mean']:.4f}, "
        f"tau={best_prior['prior_strength']:.2f}"
    )

    # ---------------------------------
    # Save raw NumPy matrix
    # ---------------------------------

    np.save(
        lambda_dir / f"{subject}_lambda.npy",
        Lambda
    )

    # ---------------------------------
    # Save readable CSV
    # ---------------------------------

    lambda_df = pd.DataFrame(
        Lambda,
        index=classes
    )

    lambda_df.index.name = "number"

    lambda_df.to_csv(
        lambda_dir / f"{subject}_lambda.csv"
    )

    lambda_metadata.append({
        "subject": subject,
        "n_presentations": X_sub.shape[0],
        "n_features": X_sub.shape[1],
        "prior_mean": best_prior["prior_mean"],
        "prior_strength": best_prior["prior_strength"],
        "alpha": alpha,
        "beta": beta,
    })


SUBJECT: YFF
Lambda shape: (9, 333)
m=0.0300, tau=10.00

SUBJECT: YFI
Lambda shape: (9, 261)
m=0.0300, tau=10.00

SUBJECT: YFJ


c:\Users\shafi\number-simplex-reproduction\.venv\Lib\site-packages\sklearn\model_selection\_split.py:1036: UserWarning: The least populated class in y has only 4 members, which is less than n_splits=5.
  warnings.warn(


Lambda shape: (9, 405)
m=0.3000, tau=10.00

SUBJECT: YFK


c:\Users\shafi\number-simplex-reproduction\.venv\Lib\site-packages\sklearn\model_selection\_split.py:1036: UserWarning: The least populated class in y has only 4 members, which is less than n_splits=5.
  warnings.warn(


Lambda shape: (9, 396)
m=0.3000, tau=10.00

SUBJECT: YFL


c:\Users\shafi\number-simplex-reproduction\.venv\Lib\site-packages\sklearn\model_selection\_split.py:1036: UserWarning: The least populated class in y has only 4 members, which is less than n_splits=5.
  warnings.warn(


Lambda shape: (9, 513)
m=0.0533, tau=24.06

SUBJECT: YFM


c:\Users\shafi\number-simplex-reproduction\.venv\Lib\site-packages\sklearn\model_selection\_split.py:1036: UserWarning: The least populated class in y has only 4 members, which is less than n_splits=5.
  warnings.warn(


Lambda shape: (9, 549)
m=0.1265, tau=10.00

SUBJECT: YFP
Lambda shape: (9, 387)
m=0.3000, tau=17.95

SUBJECT: YFR
Lambda shape: (9, 585)
m=0.3000, tau=13.40

SUBJECT: YFS
Lambda shape: (9, 531)
m=0.0300, tau=17.95

SUBJECT: YFT
Lambda shape: (9, 468)
m=0.1687, tau=24.06

SUBJECT: YFU
Lambda shape: (9, 558)
m=0.2249, tau=139.24


In [34]:
lambda_metadata_df = pd.DataFrame(
    lambda_metadata
)

lambda_metadata_df.to_csv(
    lambda_dir / "lambda_metadata.csv",
    index=False
)

display(lambda_metadata_df)

,subject,n_presentations,n_features,prior_mean,prior_strength,alpha,beta
0,YFF,109,333,0.030000,10.000000,0.300000,10.000000
1,YFI,104,261,0.030000,10.000000,0.300000,10.000000
2,YFJ,114,405,0.299962,10.000000,2.999624,10.000000
3,YFK,114,396,0.299962,10.000000,2.999624,10.000000
4,YFL,114,513,0.053347,24.057825,1.283406,24.057825
5,YFM,114,549,0.126499,10.000000,1.264990,10.000000
6,YFP,158,387,0.299962,17.954401,5.385645,17.954401
7,YFR,266,585,0.299962,13.399403,4.019317,13.399403
8,YFS,264,531,0.030000,17.954401,0.538632,17.954401
9,YFT,477,468,0.168687,24.057825,4.058231,24.057825
